In [ ]:
import pytest
from solution import Vec2, RigidBody, World

def test_vec2_operations():
    v1 = Vec2(1, 2)
    v2 = Vec2(3, 4)
    
    # Addition
    v_add = v1 + v2
    assert v_add.x == 4 and v_add.y == 6, "Vec2 addition failed"
    
    # Subtraction
    v_sub = v2 - v1
    assert v_sub.x == 2 and v_sub.y == 2, "Vec2 subtraction failed"
    
    # Scalar Multiplication (both sides if supported, but at least left)
    v_mul = v1 * 3
    assert v_mul.x == 3 and v_mul.y == 6, "Vec2 scalar multiplication failed"

def test_rigidbody_initialization_values():
    pos = Vec2(0, 0)
    vel = Vec2(1, 1)
    body = RigidBody(mass=10, inertia=100, position=pos, linear_velocity=vel, angle=0.5, angular_velocity=0.1)
    
    # Check values (not identity)
    assert body.mass == 10
    assert body.inertia == 100
    assert body.position.x == 0 and body.position.y == 0
    assert body.linear_velocity.x == 1 and body.linear_velocity.y == 1
    assert body.angle == 0.5
    assert body.angular_velocity == 0.1

def test_world_step_gravity_on_dynamic_body():
    body = RigidBody(mass=1, inertia=1, position=Vec2(0, 0), linear_velocity=Vec2(0, 0), angle=0, angular_velocity=0)
    world = World([body])
    
    dt = 0.1
    world.step(dt, 1)
    
    # Check velocity: v' = v + g*dt
    assert body.linear_velocity.x == pytest.approx(0)
    assert body.linear_velocity.y == pytest.approx(9.8 * dt, rel=1e-6, abs=1e-6)
    
    # Position must move in direction of velocity; integrator choice is free
    assert body.position.x == pytest.approx(0)
    assert body.position.y >= 0

    # No change in rotation
    assert body.angle == pytest.approx(0)
    assert body.angular_velocity == pytest.approx(0)

def test_world_step_static_body_unaffected_by_gravity():
    static_body = RigidBody(mass=0, inertia=0, position=Vec2(10, 10), linear_velocity=Vec2(0, 0), angle=0, angular_velocity=0)
    world = World([static_body])
    
    world.step(0.1, 1)
    
    # Static body should not move or accelerate
    assert static_body.position.x == 10
    assert static_body.position.y == 10
    assert static_body.linear_velocity.x == 0
    assert static_body.linear_velocity.y == 0

    inf_mass_body = RigidBody(mass=float('inf'), inertia=float('inf'), position=Vec2(20, 20), linear_velocity=Vec2(0,0), angle=0, angular_velocity=0)
    world_inf = World([inf_mass_body])
    world_inf.step(0.1, 1)

    assert inf_mass_body.position.x == 20
    assert inf_mass_body.position.y == 20
    assert inf_mass_body.linear_velocity.x == 0
    assert inf_mass_body.linear_velocity.y == 0

def test_world_with_multiple_bodies():
    b1 = RigidBody(mass=1, inertia=1, position=Vec2(0, 0), linear_velocity=Vec2(1, 0))
    b2 = RigidBody(mass=2, inertia=2, position=Vec2(10, 10), linear_velocity=Vec2(0, 0))
    b3 = RigidBody(mass=0, inertia=0, position=Vec2(20, 20), linear_velocity=Vec2(0, 0))
    world = World([b1, b2, b3])
    
    dt = 0.5
    world.step(dt, 1)
    
    # Body 1 should have moved horizontally and accelerated vertically
    assert b1.linear_velocity.x == pytest.approx(1)
    assert b1.linear_velocity.y == pytest.approx(9.8 * dt, rel=1e-6, abs=1e-6)
    assert b1.position.x > 0  # Integrator choice is free
    
    # Body 2 should only have accelerated vertically
    assert b2.linear_velocity.x == pytest.approx(0)
    assert b2.linear_velocity.y == pytest.approx(9.8 * dt, rel=1e-6, abs=1e-6)
    
    # Body 3 (static) should be unchanged
    assert b3.position.x == 20
    assert b3.position.y == 20
    assert b3.linear_velocity.y == 0

In [ ]:
import pytest
from solution import Vec2, RigidBody, Polygon, World

def create_square_body(x, y, size, mass=1):
    half = size / 2
    verts = [Vec2(-half, -half), Vec2(half, -half), Vec2(half, half), Vec2(-half, half)]
    shape = Polygon(verts)
    return RigidBody(mass=mass, inertia=1, position=Vec2(x, y), shape=shape)

def test_polygon_shape_association():
    body = create_square_body(0, 0, 2)
    assert hasattr(body, 'shape')
    assert isinstance(body.shape, Polygon)
    assert hasattr(body.shape, 'vertices')
    assert len(body.shape.vertices) == 4

def test_potential_collisions_no_overlap():
    b1 = create_square_body(0, 0, 2)
    b2 = create_square_body(5, 5, 2)
    world = World([b1, b2])
    world.step(0.01, 1)
    
    collisions = world.get_potential_collisions()
    assert len(collisions) == 0

def test_potential_collisions_simple_overlap():
    b1 = create_square_body(0, 0, 2)
    b2 = create_square_body(1, 1, 2)
    world = World([b1, b2])
    world.step(0.01, 1)
    
    collisions = world.get_potential_collisions()
    assert len(collisions) == 1
    pair = collisions[0]
    assert (pair == (b1, b2) or pair == (b2, b1))

def test_potential_collisions_touching_edge_inclusive():
    b1 = create_square_body(0, 0, 2)
    b2 = create_square_body(2, 0, 2) # AABBs touch exactly
    world = World([b1, b2])
    world.step(0.01, 1)
    
    collisions = world.get_potential_collisions()
    assert len(collisions) == 1

def test_potential_collisions_multiple_bodies():
    b1 = create_square_body(0, 0, 2) # Overlaps with b2
    b2 = create_square_body(1, 0, 2)
    b3 = create_square_body(10, 10, 2) # Isolated
    b4 = create_square_body(0.5, 0, 2) # Overlaps with b1 and b2
    world = World([b1, b2, b3, b4])
    world.step(0.01, 1)
    
    collisions = world.get_potential_collisions()
    
    # Expected pairs: (b1, b2), (b1, b4), (b2, b4)
    assert len(collisions) == 3
    
    pairs = {tuple(sorted([id(p[0]), id(p[1])])) for p in collisions}
    expected_pairs = {
        tuple(sorted([id(b1), id(b2)])),
        tuple(sorted([id(b1), id(b4)])),
        tuple(sorted([id(b2), id(b4)])),
    }
    assert pairs == expected_pairs

def test_no_self_collision():
    b1 = create_square_body(0, 0, 2)
    world = World([b1])
    world.step(0.01, 1)
    assert len(world.get_potential_collisions()) == 0

In [ ]:
import math
import pytest
from solution import Vec2, RigidBody, Polygon, get_collision_info

def create_body(verts, pos, angle=0.0):
    shape = Polygon(verts)
    body = RigidBody(mass=1, inertia=1, position=pos, shape=shape, angle=angle)
    return body

SQUARE_VERTS = [Vec2(-1, -1), Vec2(1, -1), Vec2(1, 1), Vec2(-1, 1)]

def test_no_collision():
    b1 = create_body(SQUARE_VERTS, Vec2(0, 0))
    b2 = create_body(SQUARE_VERTS, Vec2(3, 3))
    info = get_collision_info(b1, b2)
    assert info is None

def test_face_to_face_collision():
    b1 = create_body(SQUARE_VERTS, Vec2(0, 0))
    b2 = create_body(SQUARE_VERTS, Vec2(1.5, 0)) # Penetration of 0.5
    info = get_collision_info(b1, b2)
    
    assert info is not None
    assert info.depth == pytest.approx(0.5, abs=1e-3)
    # Normal should point from body1 to body2 along +x
    assert info.normal.x == pytest.approx(1.0, abs=1e-3)
    assert info.normal.y == pytest.approx(0.0, abs=1e-3)
    assert isinstance(info.contacts, list)
    assert len(info.contacts) >= 1

def test_vertex_to_face_collision_rotated_minimal_checks():
    b1 = create_body(SQUARE_VERTS, Vec2(0, 0))
    b2 = create_body(SQUARE_VERTS, Vec2(1.1, 1.1), angle=math.pi / 4)
    info = get_collision_info(b1, b2)
    
    assert info is not None
    assert info.depth > 0
    # Normal should be generally pointing from b1 to b2
    d = Vec2(b2.position.x - b1.position.x, b2.position.y - b1.position.y)
    ddot = info.normal.x * d.x + info.normal.y * d.y
    assert ddot > 0

def test_collision_info_normal_direction_symmetry():
    b1 = create_body(SQUARE_VERTS, Vec2(0, 0))
    b2 = create_body(SQUARE_VERTS, Vec2(1.5, 0))
    info = get_collision_info(b1, b2)
    info_swapped = get_collision_info(b2, b1)
    assert info is not None and info_swapped is not None
    assert info.depth == pytest.approx(info_swapped.depth, abs=1e-6)
    assert info.normal.x == pytest.approx(-info_swapped.normal.x, abs=1e-6)
    assert info.normal.y == pytest.approx(-info_swapped.normal.y, abs=1e-6)

def test_edge_to_edge_collision_exists():
    diamond_verts = [Vec2(0, -1), Vec2(1, 0), Vec2(0, 1), Vec2(-1, 0)]
    b1 = create_body(diamond_verts, Vec2(0, 0))
    b2 = create_body(diamond_verts, Vec2(1.8, 0), angle=math.pi)
    info = get_collision_info(b1, b2)

    assert info is not None
    assert info.depth > 0
    assert len(info.contacts) >= 1

In [ ]:
import pytest
from solution import Vec2, RigidBody, Polygon, World

def create_box(x, y, width, height, mass=1):
    half_w, half_h = width / 2, height / 2
    verts = [Vec2(-half_w, -half_h), Vec2(half_w, -half_h), Vec2(half_w, half_h), Vec2(-half_w, half_h)]
    shape = Polygon(verts)
    inertia = (mass / 12.0) * (width**2 + height**2) if mass > 0 else 0
    return RigidBody(mass=mass, inertia=inertia, position=Vec2(x, y), shape=shape, linear_velocity=Vec2(0,0), angle=0, angular_velocity=0)

def test_box_drops_and_resting_on_floor_inelastic():
    # Static floor at y=10, height=2 -> top surface at y=9
    floor = create_box(0, 10, 50, 2, mass=0)
    # Box center starts above the floor
    box = create_box(0, 0, 2, 2, mass=1)
    world = World([box, floor])

    # Simulate enough time for the box to fall and rest
    for _ in range(240):  # 4 seconds
        world.step(1/60, iterations=10)
    
    # Inelastic, frictionless collision: box should end up resting on the floor
    floor_surface_y = 10 - 1
    expected_center_y = floor_surface_y - 1  # box half-height
    assert box.position.y == pytest.approx(expected_center_y, abs=0.2)
    assert abs(box.linear_velocity.y) < 0.2

def test_two_boxes_colliding_head_on_inelastic():
    box1 = create_box(-5, 0, 2, 2, mass=1)
    box1.linear_velocity = Vec2(10, 0)
    box2 = create_box(5, 0, 2, 2, mass=1)
    box2.linear_velocity = Vec2(-10, 0)
    world = World([box1, box2])
    
    initial_momentum = box1.linear_velocity.x * box1.mass + box2.linear_velocity.x * box2.mass
    assert initial_momentum == pytest.approx(0)
    
    # Step until they collide and resolve
    for _ in range(60):
        world.step(1/60, iterations=5)

    # Perfectly inelastic and frictionless per Prompt 4: bodies should not separate, x-velocities should converge
    assert abs(box1.linear_velocity.x - box2.linear_velocity.x) < 1.0
    # Momentum approximately conserved (gravity contributes little in this short window)
    final_momentum = box1.linear_velocity.x * box1.mass + box2.linear_velocity.x * box2.mass
    assert final_momentum == pytest.approx(initial_momentum, abs=0.5)

def test_stacking_stability():
    floor = create_box(0, 10, 50, 2, mass=0)
    box1 = create_box(0, 8.9, 2, 2, mass=1)
    box2 = create_box(0, 6.8, 2, 2, mass=1)
    world = World([floor, box1, box2])

    for _ in range(180): # 3 seconds
        world.step(1/60, iterations=10)

    # Positions near expected resting places (tolerant)
    assert box1.position.y == pytest.approx(10 - 1 - 1, abs=0.2)
    assert box2.position.y == pytest.approx(box1.position.y - 1 - 1, abs=0.2)
    
    # Velocities near zero
    assert abs(box1.linear_velocity.y) < 0.2
    assert abs(box2.linear_velocity.y) < 0.2
    assert abs(box1.angular_velocity) < 0.5
    assert abs(box2.angular_velocity) < 0.5

In [ ]:
import pytest
from solution import Vec2, RigidBody, Polygon, World

def create_box(x, y, w, h, mass=1, restitution=0.0, static_friction=0.5, dynamic_friction=0.3):
    half_w, half_h = w / 2, h / 2
    verts = [Vec2(-half_w, -half_h), Vec2(half_w, -half_h), Vec2(half_w, half_h), Vec2(-half_w, half_h)]
    shape = Polygon(verts)
    inertia = (mass / 12.0) * (w**2 + h**2) if mass > 0 else 0
    body = RigidBody(
        mass=mass, inertia=inertia, position=Vec2(x, y), shape=shape,
        restitution=restitution, static_friction=static_friction, dynamic_friction=dynamic_friction
    )
    return body

def test_restitution_bouncy_ball():
    ball = create_box(0, -8, 2, 2, mass=1, restitution=0.8)
    floor = create_box(0, 0, 50, 2, mass=0, restitution=0.8)
    world = World([ball, floor])
    
    # Drop the ball from a height of 7 (center to surface)
    for _ in range(120):  # 2 seconds max
        world.step(1/60, 5)
        # Once it has collided with the floor it should start moving up
        if ball.position.y >= -1.0 - 1.0:
            break

    assert ball.position.y >= -2.0  # reached near the floor
    assert ball.linear_velocity.y < 0  # moving up after bounce

def test_friction_box_on_slope_static_does_not_slide():
    slope = create_box(0, 10, 30, 2, mass=0, static_friction=0.6, dynamic_friction=0.4)
    slope.angle = -0.3  # radians
    
    box = create_box(-5, 7, 2, 2, mass=1, static_friction=0.8, dynamic_friction=0.5)
    world = World([box, slope])

    # With high static friction, it should not slide appreciably
    for _ in range(120):
        world.step(1/60, 10)

    assert abs(box.linear_velocity.x) < 0.2
    assert abs(box.linear_velocity.y) < 0.5

def test_friction_sliding_to_a_stop():
    floor = create_box(0, 0, 50, 2, mass=0, dynamic_friction=0.5)
    box = create_box(-20, -1.1, 2, 2, mass=1, dynamic_friction=0.5)
    box.linear_velocity = Vec2(20, 0)
    world = World([box, floor])
    
    initial_speed = box.linear_velocity.x
    assert initial_speed > 0
    
    for _ in range(180):
        world.step(1/60, 5)

    assert box.linear_velocity.x < initial_speed / 2
    assert box.linear_velocity.x > -0.5  # should not reverse significantly

In [ ]:
import pytest
from solution import Vec2, RigidBody, Polygon, World

def create_body(verts, pos, mass=1, angle=0.0):
    shape = Polygon(verts)
    body = RigidBody(mass=mass, inertia=1, position=pos, shape=shape, angle=angle, restitution=0.1, static_friction=0.5, dynamic_friction=0.3)
    return body

def test_centroid_and_inertia_calculation():
    # A 2x4 rectangle centered at origin
    verts = [Vec2(-1, -2), Vec2(1, -2), Vec2(1, 2), Vec2(-1, 2)]
    shape = Polygon(verts)
    
    center = shape.get_centroid()
    assert center.x == pytest.approx(0)
    assert center.y == pytest.approx(0)

    mass = 12
    expected_inertia = (12 / 12.0) * (2**2 + 4**2)  # 20
    inertia = shape.get_moment_of_inertia(mass)
    assert inertia == pytest.approx(expected_inertia, rel=1e-6, abs=1e-6)

def test_off_center_collision_induces_spin():
    floor = create_body([Vec2(-50, -1), Vec2(50, -1), Vec2(50, 1), Vec2(-50, 1)], Vec2(0, 10), mass=0)
    
    # Long bar slightly rotated, one end should hit first
    bar_verts = [Vec2(-5, -0.5), Vec2(5, -0.5), Vec2(5, 0.5), Vec2(-5, 0.5)]
    bar = create_body(bar_verts, Vec2(4.5, 8.4), mass=1, angle=-0.2)
    
    world = World([bar, floor])
    assert bar.angular_velocity == pytest.approx(0)

    collided = False
    for _ in range(120):
        world.step(1/60, 10)
        if bar.position.y > 9:
            collided = True
            break
    
    assert collided, "Bar never hit the floor"
    assert bar.angular_velocity > 0.05  # gained spin

def test_body_position_is_center_of_mass_no_gravity_torque():
    # Complex polygon with off-center centroid
    verts = [Vec2(-2, -2), Vec2(2, -2), Vec2(2, 0), Vec2(0, 0), Vec2(0, 2), Vec2(-2, 2)]
    shape = Polygon(verts)

    body = RigidBody(mass=1, inertia=1, shape=shape, position=Vec2(10, 20))
    body.angular_velocity = 0
    world = World([body])
    world.step(0.1, 1)
    
    # Gravity should not induce torque on a free-falling rigid body whose position is CoM
    assert body.angular_velocity == pytest.approx(0)

In [ ]:
import pytest
from solution import Vec2, RigidBody, Polygon, World, DistanceJoint

def create_box(x, y, w, h, mass=1):
    half_w, half_h = w / 2, h / 2
    verts = [Vec2(-half_w, -half_h), Vec2(half_w, -half_h), Vec2(half_w, half_h), Vec2(-half_w, half_h)]
    shape = Polygon(verts)
    inertia = (mass / 12.0) * (w**2 + h**2) if mass > 0 else float('inf')
    body = RigidBody(mass=mass, inertia=inertia, position=Vec2(x, y), shape=shape, restitution=0, static_friction=0.5, dynamic_friction=0.3)
    return body

def test_pendulum_swing():
    pivot = create_box(0, 0, 1, 1, mass=0) # Static pivot
    bob = create_box(5, 0, 2, 2, mass=1)
    
    world = World([pivot, bob])
    
    joint = DistanceJoint(pivot, bob, Vec2(0, 0), Vec2(0, 0), 5.0)
    world.add_joint(joint)
    
    initial_x = bob.position.x
    initial_y = bob.position.y
    
    for _ in range(180): # 3 seconds
        world.step(1/60, 10)
    
    # It should have swung down (increase y)
    assert bob.position.y > initial_y - 0.01  # allow small numerical variations
    # Should not remain at initial x
    assert abs(bob.position.x - initial_x) > 0.1
    
    # The distance to the pivot should be maintained
    dx = bob.position.x - pivot.position.x
    dy = bob.position.y - pivot.position.y
    dist = (dx*dx + dy*dy) ** 0.5
    assert dist == pytest.approx(5.0, abs=0.1)

def test_two_bodies_constrained_and_pulled_apart():
    b1 = create_box(-1.5, 0, 2, 2)
    b2 = create_box(1.5, 0, 2, 2)
    world = World([b1, b2])

    joint = DistanceJoint(b1, b2, Vec2(1,0), Vec2(-1,0), 3.0)
    world.add_joint(joint)
    
    # Apply velocities that try to pull them apart
    b1.linear_velocity = Vec2(-10, 0)
    b2.linear_velocity = Vec2(10, 0)

    world.step(1/60, 10)

    dx = b1.position.x - b2.position.x
    dy = b1.position.y - b2.position.y
    dist = (dx*dx + dy*dy) ** 0.5
    assert dist == pytest.approx(3.0, abs=0.15)
    
    # Their relative velocity along the joint axis should be reduced
    assert abs(b1.linear_velocity.x - b2.linear_velocity.x) < 5.0

In [ ]:
import pytest
from solution import Vec2, RigidBody, Polygon, World

def create_box(x, y, w=2, h=2, mass=1):
    half_w, half_h = w / 2, h / 2
    verts = [Vec2(-half_w, -half_h), Vec2(half_w, -half_h), Vec2(half_w, half_h), Vec2(-half_w, half_h)]
    shape = Polygon(verts)
    inertia = (mass / 12.0) * (w**2 + h**2) if mass > 0 else float('inf')
    return RigidBody(mass=mass, inertia=inertia, position=Vec2(x, y), shape=shape, restitution=0, static_friction=0.5, dynamic_friction=0.3)

def test_stack_settles_and_sleeps():
    floor = create_box(0, 10, 50, 2, mass=0)
    boxes = [create_box(0, 9 - 2.1 * i, mass=1) for i in range(5)]
    world = World([floor] + boxes)

    # Simulate for a while to let them settle
    for _ in range(300): # 5 seconds
        world.step(1/60, 10)

    # All dynamic bodies should now be sleeping
    assert all(b.is_sleeping() for b in boxes)
    
    # And their velocities should be negligible
    assert all((b.linear_velocity.x**2 + b.linear_velocity.y**2) ** 0.5 < 0.2 for b in boxes)
    assert all(abs(b.angular_velocity) < 0.2 for b in boxes)

def test_sleeping_body_is_woken_up_by_collision():
    floor = create_box(0, 10, 50, 2, mass=0)
    box_to_sleep = create_box(0, 8.9, mass=1)
    waker_box = create_box(0.1, -50, mass=1)  # very high above; will fall later
    
    world = World([floor, box_to_sleep, waker_box])

    # Let the lower box settle and go to sleep before waker arrives
    for _ in range(120):  # 2 seconds
        world.step(1/60, 10)
    assert box_to_sleep.is_sleeping()

    # Now simulate until the waker hits the sleeping box
    woke = False
    for _ in range(300):  # up to 5 more seconds
        world.step(1/60, 10)
        if not box_to_sleep.is_sleeping():
            woke = True
            break

    assert woke, "Sleeping box was not woken by collision"
    assert (b := box_to_sleep)
    assert (b.linear_velocity.x**2 + b.linear_velocity.y**2) ** 0.5 > 0.1 or abs(b.angular_velocity) > 0.1

def test_sleeping_bodies_are_not_updated():
    floor = create_box(0, 10, 50, 2, mass=0)
    box = create_box(0, 8.9, mass=1)
    world = World([floor, box])

    # Let the box settle and sleep
    for _ in range(240):
        world.step(1/60, 10)
    assert box.is_sleeping()

    pos_before = Vec2(box.position.x, box.position.y)
    world.step(1/60, 10)
    pos_after = box.position
    
    # Sleeping box should not move
    assert pos_before.x == pytest.approx(pos_after.x, abs=1e-5)
    assert pos_before.y == pytest.approx(pos_after.y, abs=1e-5)

In [ ]:
import math
import pytest
from solution import Vec2, RigidBody, Polygon, World

def create_box(x, y, w=2, h=2, mass=1):
    half_w, half_h = w / 2, h / 2
    verts = [Vec2(-half_w, -half_h), Vec2(half_w, -half_h), Vec2(half_w, half_h), Vec2(-half_w, half_h)]
    shape = Polygon(verts)
    inertia = float('inf') if mass == 0 else (mass / 12.0) * (w**2 + h**2)
    return RigidBody(mass=mass, inertia=inertia, position=Vec2(x, y), shape=shape)

def test_raycast_miss():
    box = create_box(10, 10)
    world = World([box])
    result = world.raycast(Vec2(0, 0), Vec2(1, 0), 5.0)
    assert result is None

def test_raycast_simple_hit():
    box = create_box(10, 0)
    world = World([box])
    
    origin = Vec2(0, 0)
    direction = Vec2(1, 0)
    
    result = world.raycast(origin, direction, 20.0)
    
    assert result is not None
    assert result.body is box
    assert result.distance == pytest.approx(9.0, abs=1e-3) # box center at 10, half-width 1 -> edge at 9
    assert result.point.x == pytest.approx(9.0, abs=1e-3)
    assert result.point.y == pytest.approx(0.0, abs=1e-3)
    assert result.normal.x == pytest.approx(-1.0, abs=1e-3)
    assert result.normal.y == pytest.approx(0.0, abs=1e-3)

def test_raycast_hits_closer_body():
    box1 = create_box(10, 0)
    box2 = create_box(20, 0)
    world = World([box1, box2])
    
    result = world.raycast(Vec2(0, 0), Vec2(1, 0), 30.0)
    
    assert result is not None
    assert result.body is box1
    assert result.distance == pytest.approx(9.0, abs=1e-3)

def test_raycast_max_distance():
    box = create_box(10, 0)
    world = World([box])
    
    # This should hit
    result1 = world.raycast(Vec2(0, 0), Vec2(1, 0), 10.0)
    assert result1 is not None
    
    # This should miss due to max_distance
    result2 = world.raycast(Vec2(0, 0), Vec2(1, 0), 8.0)
    assert result2 is None

def test_raycast_from_inside_shape():
    box = create_box(0, 0)
    world = World([box])
    
    result = world.raycast(Vec2(0, 0), Vec2(1, 0), 10.0)
    
    assert result is not None
    assert result.body is box
    assert result.distance == pytest.approx(0.0, abs=1e-6)
    assert result.point.x == pytest.approx(0.0, abs=1e-6)
    assert result.point.y == pytest.approx(0.0, abs=1e-6)
    # Normal opposite to ray direction is acceptable
    assert result.normal.x == pytest.approx(-1.0, abs=1e-3)

def test_raycast_on_rotated_body():
    box = create_box(10, 10, w=4, h=2)
    box.angle = math.pi / 2  # rotated 90 degrees
    world = World([box])

    origin = Vec2(10, 0)
    direction = Vec2(0, 1)

    result = world.raycast(origin, direction, 20.0)
    
    assert result is not None
    assert result.point.x == pytest.approx(10.0, abs=1e-2)
    assert result.normal.y <= 0  # normal generally points outward against ray
    assert 0 <= result.distance <= 20.0

In [ ]:
import pytest
from solution import Vec2, RigidBody, Polygon, World

def create_box(x, y, w=2, h=2, mass=1):
    half_w, half_h = w / 2, h / 2
    verts = [Vec2(-half_w, -half_h), Vec2(half_w, -half_h), Vec2(half_w, half_h), Vec2(-half_w, half_h)]
    shape = Polygon(verts)
    inertia = (mass / 12.0) * (w**2 + h**2) if mass > 0 else float('inf')
    return RigidBody(mass=mass, inertia=inertia, position=Vec2(x, y), shape=shape, restitution=0, static_friction=0.6, dynamic_friction=0.4)

def get_total_kinetic_energy(bodies):
    energy = 0
    for b in bodies:
        if b.mass > 0 and b.inertia != float('inf'):
            v2 = b.linear_velocity.x**2 + b.linear_velocity.y**2
            energy += 0.5 * b.mass * v2
            energy += 0.5 * b.inertia * b.angular_velocity**2
    return energy

def test_stacking_stability_with_warm_starting():
    floor = create_box(0, 20, 50, 2, mass=0)
    box_height = 2.0
    num_boxes = 8  # keep reasonable for deterministic testing
    
    boxes = [
        create_box(0, 20 - 1 - (box_height/2) - (i * box_height * 1.01), w=box_height, h=box_height, mass=1) 
        for i in range(num_boxes)
    ]
    world = World([floor] + boxes)

    for _ in range(240): # 4 seconds
        world.step(1/60, 8)
    
    total_energy = get_total_kinetic_energy(boxes)
    assert total_energy < 0.5, f"Stack should be near rest, but energy is {total_energy}"
    
    top_box = boxes[-1]
    floor_surface = floor.position.y - 1
    expected_y = floor_surface - (box_height / 2) - (num_boxes - 1) * box_height
    
    assert top_box.position.y == pytest.approx(expected_y, abs=0.4)

def test_persistent_contact_no_jitter():
    slope = create_box(0, 10, 30, 2, mass=0)
    slope.angle = -0.2
    
    box = create_box(-5, 7.8, mass=1)
    world = World([box, slope])

    # Settle for 2 seconds
    for _ in range(120):
        world.step(1/60, 8)
    
    pos_after_settle = Vec2(box.position.x, box.position.y)
    
    for _ in range(120):
        world.step(1/60, 8)

    dx = box.position.x - pos_after_settle.x
    dy = box.position.y - pos_after_settle.y
    distance_drifted = (dx*dx + dy*dy) ** 0.5
    assert distance_drifted < 0.1, f"Box drifted {distance_drifted} units while supposedly at rest"